<a href="https://colab.research.google.com/github/yshzjq/Step-2.Statistical_Thinking-Machine-Learning/blob/main/Step%202.%20%ED%86%B5%EA%B3%84%EC%A0%81%20%EC%82%AC%EA%B3%A0%EC%99%80%20%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D%20%EC%9E%85%EB%AC%B8/2_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 머신러닝
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline



In [3]:
# House Prices 데이터
url ="https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"

df = pd.read_csv(url)

df.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
# 숫자형 변수만 선택
df_num = df.select_dtypes(include='number')

# 결측치 제거
df_num = df_num.dropna()

# 입력(X) / 정답(y)
X = df_num.drop(columns=['median_house_value'])
y = df_num['median_house_value']



In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



In [6]:
lr = Pipeline([
    ('scaler', StandardScaler()),# 스케일 맞추기
    ('model', LinearRegression())# 선형회귀 모델
])

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

rmse_lr

np.float64(70156.12045736385)

In [7]:
ridge = Pipeline([
    ('scaler', StandardScaler()),# 스케일 맞추기
    ('model', Ridge(alpha=0.01))# Ridge 모델
])

ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

rmse_ridge

np.float64(70156.12603169079)

In [8]:
# ridge 계수 확인
ridge_coef = pd.Series(
    ridge.named_steps['model'].coef_,
    index=X.columns
)

ridge_coef

,0
longitude,-85340.907588
latitude,-90433.862134
housing_median_age,14527.486739
total_rooms,-18173.983712
total_bedrooms,48353.330410
population,-41513.152142
households,15740.164918
median_income,76467.207658


In [9]:
lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=0.01))
])

lasso.fit(X_train, y_train)

y_pred_lasso = lasso.predict(X_test)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

rmse_lasso


np.float64(70156.12294426463)

In [10]:
# Lasso 계수 확인
lasso_coef = pd.Series(
    lasso.named_steps['model'].coef_,
    index=X.columns
)

lasso_coef


,0
longitude,-85341.619382
latitude,-90434.568148
housing_median_age,14527.417246
total_rooms,-18174.071962
total_bedrooms,48354.116666
population,-41513.164126
households,15739.453820
median_income,76467.202727


In [11]:
# 규제를 강하게 주면?
lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=3000))
])

lasso.fit(X_train, y_train)

y_pred_lasso = lasso.predict(X_test)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

rmse_lasso



np.float64(73884.28779971898)

In [13]:
# Lasso 계수 확인
lasso_coef = pd.Series(
    lasso.named_steps['model'].coef_,
    index=X.columns
)

lasso_coef

,0
longitude,-47630.368623
latitude,-52356.575069
housing_median_age,15615.243928
total_rooms,-0.000000
total_bedrooms,21739.980504
population,-15750.483485
households,0.000000
median_income,73471.455086


In [14]:
# Ridge + GridSearch
param_grid = {
'model__alpha': [0.01,0.1,1,10,100]
}

ridge_gs = GridSearchCV(
    ridge,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

ridge_gs.fit(X_train, y_train)

ridge_gs.best_params_, -ridge_gs.best_score_


({'model__alpha': 10}, np.float64(69637.68880233566))

In [16]:
ridge_gs.best_estimator_.named_steps['model'].coef_

array([-84533.18886399, -89635.25031905,  14601.99907684, -17909.5678072 ,
        47410.10182038, -41434.58639648,  16376.42068285,  76436.00167671])

In [17]:
# Lasso + GridSearch
param_grid = {
'model__alpha': [0.01,0.1,1,10,100]
}

lasso_gs = GridSearchCV(
    lasso,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

lasso_gs.fit(X_train, y_train)

lasso_gs.best_params_, -lasso_gs.best_score_


({'model__alpha': 100}, np.float64(69638.86713466966))

In [18]:
lasso_gs.best_estimator_.named_steps['model'].coef_

array([-84274.71394755, -89405.61643596,  14586.14070518, -16362.70302343,
        46474.70182749, -40886.5917249 ,  15292.00645096,  76099.69102037])